# Open-weight scale ladder — judging run

Reproduces the judging step of the paper with open weights. Everything downstream of the labels
is deterministic, so this notebook is the only part that needs a GPU.

**Protocol is fixed in `scale_prereg.py` and this notebook does not change it.** Decoding is
temperature 0 / top_p 1 / max_tokens 1024 / seed 20260908, the strict parser is the only parser,
and Qwen3 thinking mode is disabled. Every run writes a manifest recording the resolved
HuggingFace commit sha, quantisation and serving-stack versions.

Runtime: **A100 or L4**. Set it under Runtime -> Change runtime type before running anything.

In [ ]:
# 1 · repository and dependencies
REPO = "https://github.com/Taekyoon/academic_ir_research_2026.git"

import os, subprocess, sys

# Clone, or reuse an existing checkout, and FAIL LOUDLY otherwise. An earlier version of this
# cell wrote `git clone ... 2>/dev/null || echo "already cloned"`, which reported success for a
# genuine failure and then broke confusingly on the next line.
if os.path.basename(os.getcwd()) == "work":
    print("already inside the checkout:", os.getcwd())
elif os.path.isdir("work/.git"):
    os.chdir("work")
    # Reusing a checkout must also UPDATE it, otherwise a re-run silently keeps whatever
    # commit was current when the session started and any fix pushed since is invisible.
    # reset --hard touches TRACKED files only, so a fetched clef_abstracts.jsonl and any
    # labels/ output produced in this session both survive.
    for cmd in (["git", "fetch", "--depth", "1", "origin", "main"],
                ["git", "reset", "--hard", "origin/main"]):
        rr = subprocess.run(cmd, capture_output=True, text=True)
        if rr.returncode != 0:
            raise SystemExit(" ".join(cmd) + " failed: " + rr.stderr.strip())
    print("reusing and updating existing checkout:", os.getcwd())
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "work"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(f"git clone failed ({r.returncode}):\n{r.stderr.strip()}")
    os.chdir("work"); print("cloned to:", os.getcwd())

for p in ("code/scale_judge.py", "data/panel_sample.csv", "prereg/scale_prereg.py"):
    assert os.path.exists(p), f"checkout is incomplete: {p} missing"
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# BOTH versions are pinned, and transformers is the one that matters. vLLM 0.11.0 declares
# `transformers>=4.55.2` with NO upper bound, so pip leaves Colab preinstalled transformers 5.x
# in place - and transformers 5.x removed Tokenizer.all_special_tokens_extended, which vLLM
# 0.11.0 calls during tokenizer init. The run then dies with
#   AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
# after the model has already downloaded. vLLM 0.11.1 added `transformers<5` for this reason.
# Pinning transformers to the newest 4.x is the smallest fix; upgrading vLLM instead would pull
# a different pinned torch and force a multi-gigabyte reinstall.
!pip -q install "vllm==0.11.0" "transformers==4.57.6" "huggingface_hub>=0.26"

import vllm, torch, transformers
print("vllm", vllm.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())
assert transformers.__version__.startswith("4."), (
    "transformers " + transformers.__version__ + " is a 5.x release and vLLM 0.11.0 cannot use "
    "its tokenizer API. Re-run this cell, then Runtime -> Restart session, then run it again.")

## 2 · Abstracts

The repository ships PMIDs and expert labels, not abstract text — the CLEF collection itself
ships only PMIDs, and this keeps publisher-copyrighted abstracts out of the repository. Fetch
them once; the file is cached for the rest of the session.

`--email` is optional. NCBI asks automated callers to identify themselves; supply your own
address or omit the flag.

In [ ]:
!python code/fetch_abstracts.py --pmids data/pmids_panel.txt --out clef_abstracts.jsonl
!wc -l clef_abstracts.jsonl

# Expect about 2,017 records, of which roughly 7% carry a title and no abstract body. Those are
# judged on the title alone and the judging script flags them title_only.
import json
recs = [json.loads(l) for l in open("clef_abstracts.jsonl") if l.strip()]
empty = sum(1 for r in recs if not (r.get("abstract") or "").strip())
print(f"records {len(recs):,} | title-only {empty:,} ({empty/max(len(recs),1):.1%})")

## 2b · HuggingFace access for the gated families

Llama and Gemma are gated with **manual approval** on the HuggingFace API, not merely
token-gated. Two steps, both on your own account:

1. Visit each model page and accept the licence — `meta-llama/Llama-3.2-1B-Instruct`,
   `meta-llama/Llama-3.2-3B-Instruct`, `meta-llama/Llama-3.1-8B-Instruct`,
   `google/gemma-3-1b-it`, `google/gemma-3-4b-it`, `google/gemma-3-12b-it`.
   Approval is usually immediate but it is per model, not per family.
2. Put a **read** token in Colab Secrets under the name `HF_TOKEN` and enable it for this
   notebook (the key icon in the left sidebar).

The cell below reads the token from Colab Secrets so it never appears in the notebook and can
never be committed. If the secret is absent it prompts instead, with the input masked. It then
probes every gated repo and reports which are reachable — a family that is not reachable is
reported as **not attempted**, never substituted.

In [ ]:
import os, getpass
from huggingface_hub import login, HfApi

# Never write a token literal into this notebook. Colab Secrets keeps it out of the file and out
# of any commit; getpass is the fallback and masks the input.
tok = None
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    print("token source: Colab Secrets")
except Exception as e:
    print("Colab Secrets unavailable (" + type(e).__name__ + ")")
if not tok:
    tok = getpass.getpass("HF read token (input hidden, leave blank to skip gated families): ")
    print("token source: prompt")

GATED = ["meta-llama/Llama-3.2-1B-Instruct", "meta-llama/Llama-3.2-3B-Instruct",
         "meta-llama/Llama-3.1-8B-Instruct",
         "google/gemma-3-1b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it"]

if not tok:
    print("\nNo token supplied. The llama and gemma ladders will be NOT ATTEMPTED.")
    reachable = []
else:
    login(token=tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = tok          # vLLM and hub calls pick this up
    api = HfApi(token=tok)
    reachable, blocked = [], []
    for rid in GATED:
        try:
            api.model_info(rid)
            reachable.append(rid)
        except Exception as e:
            blocked.append((rid, type(e).__name__))
    print("\nreachable:")
    for r in reachable:
        print("   OK      " + r)
    for r, why in blocked:
        print("   BLOCKED " + r + "  (" + why + ")")
    if blocked:
        print("\nA BLOCKED repo means the licence has not been accepted by the account that owns")
        print("this token. Open the model page, accept, then re-run this cell. Do not work around")
        print("it with a mirror or an ungated copy - the pre-registration pins these repo ids.")

# never print or log the token itself
del tok

## 3 · Disk recovery — run this if a previous attempt filled the disk

The first attempt at the ten-arm run left about 110 GB of weights resident because the purge was
looking in the wrong place. This cell reports what is actually in the hub cache and clears it.
It touches only downloaded model weights: `labels/`, `clef_abstracts.jsonl` and the checkout are
untouched, so nothing you have already judged is lost.

Skip this cell on a fresh runtime.

In [ ]:
import os

def _free():
    st = os.statvfs("/"); return st.f_bavail * st.f_frsize / 2**30

def _scan():
    """Return the hub cache report, or None when the cache directory does not exist yet - which
    is the NORMAL state on a fresh runtime, before anything has been downloaded.

    scan_cache_dir signals that with CacheNotFound, but that class has moved between
    huggingface_hub versions (huggingface_hub.utils vs huggingface_hub.errors), so it is matched
    by name rather than imported. Anything else is re-raised: a real error must stay visible."""
    from huggingface_hub import scan_cache_dir
    try:
        return scan_cache_dir()
    except Exception as e:
        if type(e).__name__ != "CacheNotFound":
            raise
        print(f"no hub cache yet at {getattr(e, 'cache_dir', 'the default location')} - "
              f"nothing downloaded in this session")
        return None

info = _scan()
if info is None:
    print(f"disk free {_free():.0f} GB. Nothing to recover; skip to the next cell.")
else:
    print(f"hub cache at {info.size_on_disk/2**30:.1f} GB across {len(info.repos)} repos "
          f"| disk free {_free():.0f} GB")
    for r in sorted(info.repos, key=lambda x: -x.size_on_disk):
        print(f"   {r.size_on_disk/2**30:7.1f} GB  {r.repo_id}")
    if info.repos and input("\ndelete ALL cached model weights? type yes: ").strip() == "yes":
        revs = [rev.commit_hash for r in info.repos for rev in r.revisions]
        info.delete_revisions(*revs).execute()
        after = _scan()
        print(f"cleared. hub cache {0.0 if after is None else after.size_on_disk/2**30:.1f} GB "
              f"| disk free {_free():.0f} GB")
    else:
        print("\nnothing deleted")

import glob
print(f"\nlabels/ files: {len(glob.glob('labels/*'))} | "
      f"abstracts: {os.path.exists('clef_abstracts.jsonl')}")

## 4 · Smoke test before spending the GPU

64 rows on the smallest arm. Check three things in the output: `strict` is 64/64, `out_tok med`
is small (a large median means a reasoning preamble is leaking through and thinking mode is not
actually off), and `rows/s` is high enough that the full run is affordable.

In [ ]:
import os
# The abstracts file is built by the cell above and is not shipped in the repository. Check it
# here so the smoke test cannot be run out of order.
assert os.path.exists("clef_abstracts.jsonl"), (
    "clef_abstracts.jsonl is missing - run the fetch cell above first")
n = sum(1 for _ in open("clef_abstracts.jsonl"))
print(f"abstracts on disk: {n:,} records")
assert n > 1900, f"only {n} records - the fetch looks incomplete, re-run the cell above"

!python code/scale_judge.py --arm qwen3-4b --condition A --limit 64 --outdir smoke
!cat smoke/scale_manifest_qwen3-4b_A.json

## 5 · Helpers, and why the families run separately

**One family per cell, and each arm's weights are deleted as soon as both its conditions are
written.** The first attempt ran all ten arms in one cell and filled the disk: the purge globbed
`~/.cache/huggingface/hub`, which is not where the cache lives when `HF_HOME` points elsewhere,
so it silently freed nothing and about 110 GB of weights accumulated. The purge below goes
through `scan_cache_dir` / `delete_revisions`, reports the GB it actually freed, and returns -1
if the repo was not in the cache at all — a failure you can see rather than one that fills a disk.

Peak disk is therefore **one arm's weights**, not a family's total: 27.5 GB for Qwen3-14B,
15.0 GB for Llama-3.1-8B, 22.7 GB for gemma-3-12b-it. Each family cell refuses to start if the
free space is below its largest arm plus a margin.

Run this cell once, then the family cells in any order. Re-running a family cell resumes it —
arms already complete are skipped.

In [ ]:
# Shared helpers for the per-family runs. Run this once per session, after the auth cell.
import os, subprocess, sys, torch
sys.path.insert(0, "code")
from scale_judge import ARMS, WEIGHT_GB, PARAMS

VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
CAP = VRAM * 0.90

def disk_free_gb():
    st = os.statvfs("/"); return st.f_bavail * st.f_frsize / 2**30

def _scan():
    """None means the cache directory does not exist yet - the normal state before the first
    download. scan_cache_dir signals that with CacheNotFound, whose import location has moved
    between huggingface_hub versions, so it is matched by NAME and everything else is re-raised.
    Swallowing it into a NaN would print 'nan GB' and hide a real error."""
    from huggingface_hub import scan_cache_dir
    try:
        return scan_cache_dir()
    except Exception as e:
        if type(e).__name__ != "CacheNotFound":
            raise
        return None

def cache_gb():
    """Size of the hub cache through the official API rather than a guessed path. An earlier
    purge globbed ~/.cache/huggingface/hub and silently did nothing when the cache was elsewhere,
    which is how a 10-arm run filled the disk."""
    info = _scan()
    return 0.0 if info is None else info.size_on_disk / 2**30

def purge(arm):
    """Delete every cached revision of this arm's repo. Returns GB freed, or -1.0 if the repo is
    not in the cache (including the case where no cache exists at all), so a purge that frees
    nothing is visible rather than silent."""
    info = _scan()
    if info is None:
        return -1.0
    hits = [r for r in info.repos if r.repo_id == ARMS[arm]]
    if not hits:
        return -1.0
    revs = [rev.commit_hash for r in hits for rev in r.revisions]
    before = info.size_on_disk
    info.delete_revisions(*revs).execute()
    after = _scan()
    return (before - (0 if after is None else after.size_on_disk)) / 2**30

def done(arm, cond, harness="guided"):
    suf = {"free": "", "guided": "_guided", "guided_strict": "_guidedstrict"}[harness]
    p = f"labels/scale_labels_{arm}_{cond}{suf}.jsonl"
    return os.path.exists(p) and sum(1 for ln in open(p) if ln.strip()) >= 2025

def reachable_arm(arm, granted):
    if WEIGHT_GB[arm] > CAP:
        return False, f"bf16 {WEIGHT_GB[arm]} GB exceeds {CAP:.0f} GB VRAM"
    if not arm.startswith("qwen3") and ARMS[arm].split("/")[-1].lower() not in granted:
        return False, "gated access not granted"
    return True, ""

def run_family(fam, arms, granted, harness="guided"):
    """One family, one call. Each arm's weights are purged as soon as BOTH its conditions are
    written, so peak disk is one arm's weights rather than the family's total. A run that stops
    part-way leaves complete arms on disk and is resumed by calling this again."""
    plan, skip = [], []
    for a in arms:
        good, why = reachable_arm(a, granted)
        (plan.append(a) if good else skip.append((a, why)))
    need = max([WEIGHT_GB[a] for a in plan], default=0) + 8      # weights + margin
    print(f"{fam}: {len(plan)} arms, harness={harness}  {plan}")
    for a, why in skip:
        print(f"   NOT ATTEMPTED  {a}  ({why})")
    print(f"   disk free {disk_free_gb():.0f} GB | hub cache {cache_gb():.0f} GB | "
          f"largest arm needs ~{need:.0f} GB")
    if disk_free_gb() < need:
        raise SystemExit(f"only {disk_free_gb():.0f} GB free but the largest arm needs ~{need:.0f} "
                         f"GB. Purge the cache or restart the runtime before continuing.")
    ok_, bad = [], []
    for arm in plan:
        for cond in ("A", "C"):
            tag = f"{arm}_{cond}_{harness}"
            if done(arm, cond, harness):
                print(f"   {tag}: already complete, skipping", flush=True); ok_.append(tag); continue
            print(f"\n=== {arm} / condition {cond}  (disk free {disk_free_gb():.0f} GB)", flush=True)
            cmd = ["python", "code/scale_judge.py", "--arm", arm, "--condition", cond]
            if harness == "guided": cmd.append("--guided")
            elif harness == "guided_strict": cmd.append("--guided-strict")
            r = subprocess.run(cmd)
            (ok_ if r.returncode == 0 and done(arm, cond, harness) else bad).append(tag)
            if tag in bad:
                print(f"   {tag} did not complete - NOT quantised, NOT substituted", flush=True)
        if all(done(arm, c, harness) for c in ("A", "C")):
            freed = purge(arm)
            print(f"   purged {arm}: freed {freed:.1f} GB | disk free {disk_free_gb():.0f} GB",
                  flush=True)
    print(f"\n{fam} completed {len(ok_)}/{2*len(plan)}: {ok_}")
    if bad: print(f"{fam} FAILED: {bad}")
    return ok_, bad

granted = {r.split("/")[-1].lower() for r in reachable} if "reachable" in dir() else set()
print(f"{torch.cuda.get_device_name(0)} | {VRAM:.0f} GB VRAM | usable {CAP:.0f} GB")
print(f"disk free {disk_free_gb():.0f} GB | hub cache {cache_gb():.0f} GB")
print(f"gated repos granted: {len(granted)}")

## 6 · qwen3 — permissive guided

4 arms x 2 conditions. Largest arm 27.5 GB in bf16.

**Permissive guided decoding**: `[\s\S]*## final score: [0-3]`. The model may reason, but
constrained decoding will not let it stop before emitting a valid score line, so the parse rate
should be 1.00 and every arm enters the verdicts. Per amendment 5 this is the primary basis for
all open arms — a remedy applied only to the arms that failed would itself be a selection effect.

Safe to re-run: completed arms are skipped. Guided labels are written to
`scale_labels_<arm>_<cond>_guided.jsonl`, so run 2's free labels are not overwritten.

In [ ]:
ok_qwen3, bad_qwen3 = run_family("qwen3", ['qwen3-1.7b', 'qwen3-4b', 'qwen3-8b', 'qwen3-14b'], granted, harness="guided")


## 7 · llama — permissive guided

3 arms x 2 conditions. Largest arm 15.0 GB in bf16.

**Permissive guided decoding**: `[\s\S]*## final score: [0-3]`. The model may reason, but
constrained decoding will not let it stop before emitting a valid score line, so the parse rate
should be 1.00 and every arm enters the verdicts. Per amendment 5 this is the primary basis for
all open arms — a remedy applied only to the arms that failed would itself be a selection effect.

Safe to re-run: completed arms are skipped. Guided labels are written to
`scale_labels_<arm>_<cond>_guided.jsonl`, so run 2's free labels are not overwritten.

In [ ]:
ok_llama, bad_llama = run_family("llama", ['llama-3.2-1b-instruct', 'llama-3.2-3b-instruct', 'llama-3.1-8b-instruct'], granted, harness="guided")


## 8 · gemma — permissive guided

3 arms x 2 conditions. Largest arm 22.7 GB in bf16.

**Permissive guided decoding**: `[\s\S]*## final score: [0-3]`. The model may reason, but
constrained decoding will not let it stop before emitting a valid score line, so the parse rate
should be 1.00 and every arm enters the verdicts. Per amendment 5 this is the primary basis for
all open arms — a remedy applied only to the arms that failed would itself be a selection effect.

Safe to re-run: completed arms are skipped. Guided labels are written to
`scale_labels_<arm>_<cond>_guided.jsonl`, so run 2's free labels are not overwritten.

In [ ]:
ok_gemma, bad_gemma = run_family("gemma", ['gemma-3-1b-it', 'gemma-3-4b-it', 'gemma-3-12b-it'], granted, harness="guided")


## 9 · Harness check — the four arms that already passed free generation

qwen3-1.7b, qwen3-8b, qwen3-14b and gemma-3-12b-it parsed at 1.00 without guidance in run 2, so
their free labels already exist. The cell above has now judged them guided as well, giving paired
free/guided labels on identical pairs.

That pairing is registered as **H-H1** and it gates the paper's open-versus-proprietary claim:
the proprietary arms were judged through APIs, which offer no constrained decoding, so they can
never be re-run guided. Uniformity is achievable within the open arms only. H-H1 measures how far
the two harnesses disagree, and that measurement is the bound we are allowed to quote when
comparing a guided open arm to a free proprietary one.

Nothing to run here — this cell just confirms both harnesses are present for those four arms
before you download.

In [ ]:
PAIRED = ["qwen3-1.7b", "qwen3-8b", "qwen3-14b", "gemma-3-12b-it"]
print("arm                     cond   free  guided")
for arm in PAIRED:
    for cond in ("A", "C"):
        print(f"  {arm:22s} {cond:4s} {str(done(arm, cond, 'free')):>6s} "
              f"{str(done(arm, cond, 'guided')):>7s}")
missing = [(a, c, h) for a in PAIRED for c in ("A", "C") for h in ("free", "guided")
           if not done(a, c, h)]
print(f"\nmissing for H-H1: {missing if missing else 'none - the pairing is complete'}")

## 10 · Collect

Label files are small — about 150 KB per arm-condition, so the whole set is a couple of MB. The
zip is written to `/content`, not into the checkout, and the cell reports free space first
because the earlier attempt failed here with `No space left on device` while the weights were
still resident.

If a family is incomplete, download anyway and say which arms completed. A partial ladder is
reported as partial, never presented as the registered design.

In [ ]:
import glob, os
print(f"disk free {disk_free_gb():.0f} GB | hub cache {cache_gb():.0f} GB")
labs = sorted(glob.glob("labels/scale_labels_*.jsonl"))
mans = sorted(glob.glob("labels/scale_manifest_*.json"))
tot = sum(os.path.getsize(f) for f in labs + mans) / 2**20
print(f"label files {len(labs)} | manifests {len(mans)} | {tot:.1f} MB total\n")
for f in labs:
    n = sum(1 for ln in open(f) if ln.strip())
    print(f"  {os.path.basename(f):46s} {n:5d} rows")
!cd /content/work && zip -qr /content/scale_labels.zip labels/scale_labels_*.jsonl labels/scale_manifest_*.json
print(f"\nzip {os.path.getsize('/content/scale_labels.zip')/2**20:.1f} MB")
from google.colab import files
files.download("/content/scale_labels.zip")